In [1]:
import pandas as pd
import numpy as np

from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Examination"
)

xpt_file = DATA_DIR / "BMX_G.xpt"
csv_file = DATA_DIR / "BMX_G_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched source values.
#
source_df = df.copy()


print("=" * 80)
print("NHANES 2011-2012 BMX_G XPT -> CSV")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "BMDSTATS",
    "BMXWT",
    "BMIWT",
    "BMXRECUM",
    "BMIRECUM",
    "BMXHEAD",
    "BMIHEAD",
    "BMXHT",
    "BMIHT",
    "BMXBMI",
    "BMDBMIC",
    "BMXLEG",
    "BMILEG",
    "BMXARML",
    "BMIARML",
    "BMXARMC",
    "BMIARMC",
    "BMXWAIST",
    "BMIWAIST",
    "BMXSAD1",
    "BMXSAD2",
    "BMXSAD3",
    "BMXSAD4",
    "BMDAVSAD",
    "BMDSADCM"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "BMX_G column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 9338:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 26:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,338 rows x 26 columns."
)


# ============================================================
# 4. DEFINE GENUINE DECIMAL MEASUREMENT VARIABLES
# ============================================================

#
# CRITICAL:
#
# These 14 fields contain genuine anthropometric decimals.
#
# DO NOT:
#
#   - convert them to Int64
#   - round them merely to remove .0
#   - specify float_format on export
#
# Examples:
#
#     BMXWT     86.4
#     BMXHT    160.0
#     BMXBMI    33.8
#     BMXWAIST  92.5
#
# must remain measurement values.
#

DECIMAL_COLUMNS = [
    "BMXWT",
    "BMXRECUM",
    "BMXHEAD",
    "BMXHT",
    "BMXBMI",
    "BMXLEG",
    "BMXARML",
    "BMXARMC",
    "BMXWAIST",
    "BMXSAD1",
    "BMXSAD2",
    "BMXSAD3",
    "BMXSAD4",
    "BMDAVSAD"
]


INTEGER_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if col not in DECIMAL_COLUMNS
]


if len(DECIMAL_COLUMNS) != 14:

    raise ValueError(
        "Expected exactly 14 genuine decimal variables."
    )


if len(INTEGER_COLUMNS) != 12:

    raise ValueError(
        "Expected exactly 12 whole-number variables."
    )


print("\n--- VARIABLE TYPE PLAN ---")

print(
    "Decimal measurement variables:",
    len(DECIMAL_COLUMNS)
)

print(
    "Whole-number identifier/code variables:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# Attached BMX_G.xpt:
#
#     tiny artifacts = 0
#
# Therefore NO zero restoration is needed.
#
# If a future/local copy differs, STOP rather than
# automatically changing values.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)

print(
    "Tiny XPORT values:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in BMX_G. Do not automatically "
        "replace them with zero."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 6. DETECT GENUINE FRACTIONAL VARIABLES
# ============================================================

#
# This proves that ONLY the expected measurement
# variables contain fractional values.
#

print("\n--- GENUINE DECIMAL CHECK ---")


detected_decimal_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        detected_decimal_columns.append(
            col
        )


        print(
            f"{col}: "
            f"{int(fractional_mask.sum()):,} "
            "fractional observations"
        )


if detected_decimal_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Unexpected decimal-variable pattern.\n"
        f"Expected: {DECIMAL_COLUMNS}\n"
        f"Actual:   {detected_decimal_columns}"
    )


print(
    "\nPASS: Exactly 14 genuine decimal "
    "measurement variables confirmed."
)


# ============================================================
# 7. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if not (
    df["SEQN"].min() == 62161
    and
    df["SEQN"].max() == 71916
    and
    int(
        df["SEQN"]
        .isna()
        .sum()
    ) == 0
    and
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 8. FORMAT ONLY PROVEN WHOLE-NUMBER COLUMNS
# ============================================================

#
# These include:
#
# SEQN
# BMDSTATS
# BMIWT
# BMIRECUM
# BMIHEAD
# BMIHT
# BMDBMIC
# BMILEG
# BMIARML
# BMIARMC
# BMIWAIST
# BMDSADCM
#
# NO .round() is used.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "genuine decimal values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: 12 whole-number fields converted "
    "to nullable Int64."
)

print(
    "PASS: 14 body-measurement fields remain decimal."
)


# ============================================================
# 9. ATTACHED XPT - COMPONENT STATUS CHECK
# ============================================================

#
# Exact frequency pattern in attached BMX_G.xpt.
#

EXPECTED_BMDSTATS = {
    1: 8520,
    2: 310,
    3: 433,
    4: 75
}


actual = (
    df["BMDSTATS"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual != EXPECTED_BMDSTATS:

    raise ValueError(
        "BMDSTATS frequencies changed.\n"
        f"Expected: {EXPECTED_BMDSTATS}\n"
        f"Actual:   {actual}"
    )


if int(
    df["BMDSTATS"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Unexpected BMDSTATS missing values."
    )


print(
    "PASS: BMDSTATS matches attached XPT."
)


# ============================================================
# 10. ATTACHED XPT - COMMENT CODE CHECKS
# ============================================================

COMMENT_CHECKS = {

    "BMIWT": {
        "values": {
            1: 15,
            3: 348,
            4: 16
        },
        "missing": 8959
    },

    "BMIRECUM": {
        "values": {
            1: 31
        },
        "missing": 9307
    },

    "BMIHEAD": {
        "values": {},
        "missing": 9338
    },

    "BMIHT": {
        "values": {
            1: 56,
            3: 212
        },
        "missing": 9070
    },

    "BMILEG": {
        "values": {
            1: 354
        },
        "missing": 8984
    },

    "BMIARML": {
        "values": {
            1: 369
        },
        "missing": 8969
    },

    "BMIARMC": {
        "values": {
            1: 373
        },
        "missing": 8965
    },

    "BMIWAIST": {
        "values": {
            1: 456
        },
        "missing": 8882
    }
}


print("\n--- COMMENT CODE CHECKS ---")


for col, expected in COMMENT_CHECKS.items():

    actual_values = (
        df[col]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if actual_values != expected["values"]:

        raise ValueError(
            f"{col} values changed.\n"
            f"Expected: {expected['values']}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected["missing"]:

        raise ValueError(
            f"{col} missing count changed."
        )


    print(
        f"PASS: {col}"
    )


# ============================================================
# 11. ATTACHED XPT - BMI CATEGORY CHECK
# ============================================================

#
# CDC:
#
# BMDBMIC:
#
# 1 = Underweight
# 2 = Normal weight
# 3 = Overweight
# 4 = Obese
#
# It is created for children/adolescents age 2-19.
#

EXPECTED_BMDBMIC = {
    1: 126,
    2: 2155,
    3: 481,
    4: 593
}


actual = (
    df["BMDBMIC"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual != EXPECTED_BMDBMIC:

    raise ValueError(
        "BMDBMIC frequency pattern changed."
    )


if int(
    df["BMDBMIC"]
    .isna()
    .sum()
) != 5983:

    raise ValueError(
        "BMDBMIC missing count changed."
    )


print(
    "PASS: BMDBMIC matches attached XPT."
)


# ============================================================
# 12. ATTACHED XPT - SAD COMMENT CHECK
# ============================================================

#
# CDC defines BMDSADCM codes 1-5.
#

EXPECTED_BMDSADCM = {
    1: 427,
    2: 19,
    3: 7,
    4: 22,
    5: 10
}


actual = (
    df["BMDSADCM"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual != EXPECTED_BMDSADCM:

    raise ValueError(
        "BMDSADCM frequency pattern changed."
    )


if int(
    df["BMDSADCM"]
    .isna()
    .sum()
) != 8853:

    raise ValueError(
        "BMDSADCM missing count changed."
    )


print(
    "PASS: BMDSADCM matches attached XPT."
)


# ============================================================
# 13. MEASUREMENT COUNT / RANGE CHECKS
# ============================================================

#
# Exact ranges/counts from attached BMX_G.xpt.
#
# These checks are useful for detecting:
#
#   truncation
#   accidental rounding
#   missing-value changes
#   column corruption
#

MEASUREMENT_CHECKS = {

    "BMXWT": {
        "n": 9243,
        "missing": 95,
        "min": 3.6,
        "max": 216.1
    },

    "BMXRECUM": {
        "n": 1079,
        "missing": 8259,
        "min": 48.3,
        "max": 115.6
    },

    "BMXHEAD": {
        "n": 236,
        "missing": 9102,
        "min": 34.6,
        "max": 48.4
    },

    "BMXHT": {
        "n": 8615,
        "missing": 723,
        "min": 82.0,
        "max": 204.5
    },

    "BMXBMI": {
        "n": 8602,
        "missing": 736,
        "min": 12.4,
        "max": 82.1
    },

    "BMXLEG": {
        "n": 6955,
        "missing": 2383,
        "min": 24.8,
        "max": 52.8
    },

    "BMXARML": {
        "n": 8826,
        "missing": 512,
        "min": 10.0,
        "max": 48.1
    },

    "BMXARMC": {
        "n": 8826,
        "missing": 512,
        "min": 10.5,
        "max": 58.1
    },

    "BMXWAIST": {
        "n": 8204,
        "missing": 1134,
        "min": 38.7,
        "max": 176.0
    },

    "BMXSAD1": {
        "n": 6795,
        "missing": 2543,
        "min": 10.0,
        "max": 40.4
    },

    "BMXSAD2": {
        "n": 6795,
        "missing": 2543,
        "min": 9.9,
        "max": 40.8
    },

    "BMXSAD3": {
        "n": 398,
        "missing": 8940,
        "min": 11.0,
        "max": 36.4
    },

    "BMXSAD4": {
        "n": 398,
        "missing": 8940,
        "min": 11.1,
        "max": 36.4
    },

    "BMDAVSAD": {
        "n": 6795,
        "missing": 2543,
        "min": 10.0,
        "max": 40.6
    }
}


print("\n--- MEASUREMENT RANGE CHECKS ---")


for col, expected in MEASUREMENT_CHECKS.items():

    s = df[col]


    if int(
        s.notna()
        .sum()
    ) != expected["n"]:

        raise ValueError(
            f"{col}: non-missing count changed."
        )


    if int(
        s.isna()
        .sum()
    ) != expected["missing"]:

        raise ValueError(
            f"{col}: missing count changed."
        )


    if not np.isclose(
        s.min(),
        expected["min"],
        rtol=0,
        atol=0
    ):

        raise ValueError(
            f"{col}: minimum changed."
        )


    if not np.isclose(
        s.max(),
        expected["max"],
        rtol=0,
        atol=0
    ):

        raise ValueError(
            f"{col}: maximum changed."
        )


    print(
        f"PASS: {col}"
    )


# ============================================================
# 14. CDC HARD-EDIT RANGE CHECKS
# ============================================================

#
# CDC publishes these hard-edit limits:
#
# BMXWT:
#     0 - 440 kg
#
# BMXRECUM:
#     16 - 119.3 cm
#
# BMXHEAD:
#     0 - 99.9999 cm
#
# BMXHT:
#     70 - 208.4 cm
#
# BMXLEG:
#     0 - 200 cm
#
# BMXARML:
#     0 - 200 cm
#
# BMXARMC:
#     0 - 200 cm
#
# BMXWAIST:
#     0 - 200 cm
#

CDC_HARD_EDITS = {

    "BMXWT": (
        0,
        440
    ),

    "BMXRECUM": (
        16,
        119.3
    ),

    "BMXHEAD": (
        0,
        99.9999
    ),

    "BMXHT": (
        70,
        208.4
    ),

    "BMXLEG": (
        0,
        200
    ),

    "BMXARML": (
        0,
        200
    ),

    "BMXARMC": (
        0,
        200
    ),

    "BMXWAIST": (
        0,
        200
    )
}


print("\n--- CDC HARD-EDIT CHECKS ---")


for col, (
    minimum,
    maximum
) in CDC_HARD_EDITS.items():

    s = (
        df[col]
        .dropna()
    )


    if not s.between(
        minimum,
        maximum
    ).all():

        raise ValueError(
            f"{col}: value outside CDC "
            f"hard-edit range {minimum}-{maximum}."
        )


    print(
        f"PASS: {col} within CDC hard-edit range."
    )


# ============================================================
# 15. VERIFY BMI CALCULATION
# ============================================================

#
# CDC states:
#
# BMI =
#
#      weight in kg
#      -----------------
#      height in meters²
#
# then rounded to ONE decimal place.
#
#
# IMPORTANT:
#
# Python's standard round() uses banker's rounding.
#
# CDC/SAS behavior at half boundaries can differ.
#
# Example from attached BMX_G:
#
# Weight = 86.4 kg
# Height = 160.0 cm
#
# Exact BMI = 33.75
#
# Released BMXBMI = 33.8
#
# Therefore use Decimal + ROUND_HALF_UP for validation.
#

def bmi_half_up(
    weight_kg,
    height_cm
):

    weight = Decimal(
        str(weight_kg)
    )

    height_m = (
        Decimal(
            str(height_cm)
        )
        /
        Decimal("100")
    )


    bmi = (
        weight
        /
        (
            height_m
            *
            height_m
        )
    )


    return float(
        bmi.quantize(
            Decimal("0.1"),
            rounding=ROUND_HALF_UP
        )
    )


bmi_rows = (
    df[
        [
            "BMXWT",
            "BMXHT",
            "BMXBMI"
        ]
    ]
    .dropna()
)


bmi_expected = bmi_rows.apply(
    lambda row:
        bmi_half_up(
            row["BMXWT"],
            row["BMXHT"]
        ),
    axis=1
)


bmi_actual = (
    bmi_rows["BMXBMI"]
    .to_numpy(
        dtype=float
    )
)


bmi_mismatches = int(
    np.sum(
        bmi_expected.to_numpy()
        !=
        bmi_actual
    )
)


print("\n--- BMI FORMULA CHECK ---")

print(
    "Rows checked:",
    f"{len(bmi_rows):,}"
)

print(
    "BMI mismatches:",
    bmi_mismatches
)


if bmi_mismatches != 0:

    raise ValueError(
        "BMXBMI does not match CDC BMI "
        "calculation for one or more records."
    )


print(
    "PASS: All available BMXBMI values match "
    "CDC weight/height calculation."
)


# ============================================================
# 16. VERIFY AVERAGE SAD
# ============================================================

#
# CDC algorithm:
#
# If only SAD1 and SAD2 are available:
#
#     mean(SAD1, SAD2)
#
# rounded to 0.1 cm.
#
# If four readings exist:
#
#     determine the three closest readings;
#     if a unique group exists, use their mean;
#     otherwise use mean of all four.
#
# CDC/SAS rounds to 0.1.
#

SAD_COLUMNS = [
    "BMXSAD1",
    "BMXSAD2",
    "BMXSAD3",
    "BMXSAD4"
]


def round_tenth_half_up(
    value
):

    if not isinstance(
        value,
        Decimal
    ):

        value = Decimal(
            str(value)
        )


    return value.quantize(
        Decimal("0.1"),
        rounding=ROUND_HALF_UP
    )


def calculate_expected_sad(
    row
):

    values = [
        row[col]
        for col in SAD_COLUMNS
        if pd.notna(
            row[col]
        )
    ]


    if len(values) == 0:

        return np.nan


    values = [
        Decimal(
            str(v)
        )
        for v in values
    ]


    # -------------------------
    # TWO READINGS
    # -------------------------

    if len(values) == 2:

        mean_value = (
            sum(values)
            /
            Decimal("2")
        )


        return float(
            round_tenth_half_up(
                mean_value
            )
        )


    # -------------------------
    # FOUR READINGS
    # -------------------------

    if len(values) == 4:

        triplets = [
            (0, 1, 2),
            (0, 1, 3),
            (0, 2, 3),
            (1, 2, 3)
        ]


        ranges = []


        for indexes in triplets:

            selected = [
                values[i]
                for i in indexes
            ]


            range_value = (
                max(selected)
                -
                min(selected)
            )


            ranges.append(
                round_tenth_half_up(
                    range_value
                )
            )


        ordered = sorted(
            ranges
        )


        #
        # Tie for smallest range:
        #
        # use all four measurements.
        #

        if ordered[0] == ordered[1]:

            mean_value = (
                sum(values)
                /
                Decimal("4")
            )


            return float(
                round_tenth_half_up(
                    mean_value
                )
            )


        #
        # Unique closest triplet.
        #

        selected_index = (
            ranges.index(
                ordered[0]
            )
        )


        indexes = (
            triplets[
                selected_index
            ]
        )


        selected = [
            values[i]
            for i in indexes
        ]


        mean_value = (
            sum(selected)
            /
            Decimal("3")
        )


        return float(
            round_tenth_half_up(
                mean_value
            )
        )


    raise ValueError(
        "Unexpected number of available "
        "SAD measurements."
    )


sad_nonmissing_counts = (
    df[SAD_COLUMNS]
    .notna()
    .sum(
        axis=1
    )
)


print("\n--- SAD READING PATTERN ---")

print(
    sad_nonmissing_counts
    .value_counts()
    .sort_index()
    .to_dict()
)


EXPECTED_SAD_PATTERN = {
    0: 2543,
    2: 6397,
    4: 398
}


if (
    sad_nonmissing_counts
    .value_counts()
    .sort_index()
    .to_dict()
    != EXPECTED_SAD_PATTERN
):

    raise ValueError(
        "Unexpected SAD measurement pattern."
    )


sad_expected = df.apply(
    calculate_expected_sad,
    axis=1
)


sad_actual = (
    df["BMDAVSAD"]
    .to_numpy(
        dtype=float
    )
)


sad_mismatches = int(
    np.sum(
        ~np.isclose(
            sad_expected.to_numpy(
                dtype=float
            ),
            sad_actual,
            rtol=0,
            atol=0,
            equal_nan=True
        )
    )
)


print(
    "BMDAVSAD mismatches:",
    sad_mismatches
)


if sad_mismatches != 0:

    raise ValueError(
        "BMDAVSAD does not match the "
        "CDC SAD calculation algorithm."
    )


print(
    "PASS: All BMDAVSAD values match "
    "CDC calculation logic."
)


# ============================================================
# 17. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# BMX_G requires ZERO numeric corrections.
#
# Therefore all numeric values must remain exactly
# equal to the source XPT.
#
# Only storage representation changes for code fields:
#
#     SEQN
#
#     62161.0
#        ->
#     62161
#
# Measurement values are untouched.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 26 variables preserve "
    "source XPT values exactly."
)


# ============================================================
# 18. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO float_format
# NO rounding
#
# This preserves body-measurement decimals naturally.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 19. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 20. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- EXACT XPT -> CSV FIDELITY ---")


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 26 CSV columns round-trip "
    "to exact XPT numeric values."
)


# ============================================================
# 21. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 22. READ RAW CSV AS TEXT
# ============================================================

#
# Numeric fidelity was verified above.
#
# Read as text now to inspect formatting.
#

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 23. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    numeric_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        numeric_values
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 24. .0 CHECK - INTEGER FIELDS ONLY
# ============================================================

#
# VERY IMPORTANT:
#
# DO NOT search the entire BMX CSV for ".0".
#
# Measurement fields legitimately contain values such as:
#
#     BMXHT      160.0
#     BMXARML     30.0
#     BMDAVSAD    22.0
#
# Only identifier/status/comment fields should have
# unnecessary ".0" removed.
#

print("\n--- INTEGER COLUMN .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = (
        raw_csv_df[col]
    )


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected .0 formatting in integer fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary .0 "
    "in identifier/code fields."
)


# ============================================================
# 25. DECIMAL .0 VALUES ARE ALLOWED
# ============================================================

#
# Informational only.
#
# Genuine body-measurement fields may legitimately
# serialize with ".0".
#

measurement_dot_zero_counts = {}


for col in DECIMAL_COLUMNS:

    count = int(
        (
            raw_csv_df[col].ne("")
            &
            raw_csv_df[col]
            .str.endswith(".0")
        )
        .sum()
    )


    if count > 0:

        measurement_dot_zero_counts[col] = count


print(
    "\nMeasurement variables containing "
    "legitimate '.0' values:",
    len(
        measurement_dot_zero_counts
    )
)


print(
    "PASS: Genuine measurement .0 values "
    "were left AS IS."
)


# ============================================================
# 26. EXACT DECIMAL MEASUREMENT FIDELITY
# ============================================================

print("\n--- MEASUREMENT DECIMAL FIDELITY ---")


decimal_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        decimal_failures.append(
            col
        )


if decimal_failures:

    raise ValueError(
        "Measurement decimal fidelity failed:\n"
        + str(decimal_failures)
    )


print(
    "PASS: All 14 genuine decimal "
    "measurement variables preserved exactly."
)


# ============================================================
# 27. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 28. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if not (
    final_seqn.min() == 62161
    and
    final_seqn.max() == 71916
    and
    int(
        final_seqn.duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains .0."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no .0."
)


# ============================================================
# 29. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 80
)


print(
    "BMX_G CONVERSION COMPLETE"
)


print(
    "=" * 80
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Validated zero restorations: NONE"
)


print(
    "Genuine decimal measurement variables:",
    len(DECIMAL_COLUMNS)
)


print(
    "Whole-number identifier/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary .0 in integer fields: "
    "0 expected"
)


print(
    "Measurement decimals altered: "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2011-2012 BMX_G XPT -> CSV
Rows: 9,338
Columns: 26
PASS: Structure = 9,338 rows x 26 columns.

--- VARIABLE TYPE PLAN ---
Decimal measurement variables: 14
Whole-number identifier/code variables: 12

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros: 0
Tiny XPORT values: 0
PASS: No tiny XPORT artifacts.

--- GENUINE DECIMAL CHECK ---
BMXWT: 8,333 fractional observations
BMXRECUM: 964 fractional observations
BMXHEAD: 216 fractional observations
BMXHT: 7,773 fractional observations
BMXBMI: 7,698 fractional observations
BMXLEG: 5,590 fractional observations
BMXARML: 6,639 fractional observations
BMXARMC: 7,516 fractional observations
BMXWAIST: 6,776 fractional observations
BMXSAD1: 5,785 fractional observations
BMXSAD2: 5,778 fractional observations
BMXSAD3: 336 fractional observations
BMXSAD4: 344 fractional observations
BMDAVSAD: 6,142 fractional observations

PASS: Exactly 14 genuine decimal measurement variables confirmed.

--- SEQN ---
Minimum: 62161
Maximum: 71916
Missin